# Assignment Week 2

## Workflow: Weekly Meal Planner & Grocery List

### Role 1: User Requirement Analyzer

Input:
- Diet preference
- Number of people
- Cuisine preference

Output:
- Structured meal requirements

### Role 2: Meal Planner

Input:
- Meal requirements

Output:
- 7-day meal plan

### Role 3: Grocery List Generator

Input:
- 7-day meal plan

Output:
- Combined grocery list

### Role 4: Nutrition & Variety Reviewer

Input:
- Meal plan
- Grocery list

Output:
- PASS or FAIL review

### Review Loop

Meal Planner → Grocery List Generator → Reviewer

If review fails:
Reviewer sends work back to Meal Planner.

Maximum 3 review rounds.

# Success Criteria

1. All meals follow the selected diet.

2. No meal repeats within 3 days.

3. Meal portions match the number of people.

4. Grocery list combines duplicate ingredients.

5. The plan contains meals for all 7 days.

In [1]:
!pip install openai

In [2]:
from openai import OpenAI

client = OpenAI(
    api_key="YOUR_API_KEY"
)

In [3]:
def chat(prompt):

    response = client.responses.create(
        model="gpt-5",
        input=prompt
    )

    return response.output_text

In [5]:
def analyze_requirements(diet, people, cuisine):

    prompt = f"""
    
    Analyze the meal planning request.

    Diet: {diet}
    Number of People: {people}
    Cuisine: {cuisine}

    Return structured requirements.
    
    """

    return chat(prompt)

In [6]:
def create_meal_plan(requirements):

    prompt = f"""
    
    Create a 7-day meal plan.

    Requirements:
    {requirements}

    Include:
    - Breakfast
    - Lunch
    - Dinner

    Avoid repeating meals.
    
    """

    return chat(prompt)

In [7]:
def create_grocery_list(meal_plan):

    prompt = f"""
    
    Create a grocery list from this meal plan.

    Merge duplicate ingredients.

    Meal Plan:
    {meal_plan}
    
    """

    return chat(prompt)

In [8]:
def review_plan(meal_plan, grocery_list):

    prompt = f"""
    
    Check the following criteria:

    1. Meals follow the diet.
    2. No meal repeats within 3 days.
    3. Covers all 7 days.
    4. Grocery list merges duplicates.
    5. Portions fit the number of people.

    Return:

    PASS or FAIL

    Explain issues.

    Meal Plan:
    {meal_plan}

    Grocery List:
    {grocery_list}
    
    """

    return chat(prompt)

In [11]:
def run_workflow(diet, people, cuisine):

    requirements = analyze_requirements(
        diet,
        people,
        cuisine
    )

    rounds = 0

    while rounds < 3:

        meal_plan = create_meal_plan(
            requirements
        )

        grocery_list = create_grocery_list(
            meal_plan
        )

        review = review_plan(
            meal_plan,
            grocery_list
        )

        if "PASS" in review.upper():
            break

        rounds += 1

    timestamp = datetime.now().strftime()

    folder = f"outputs/meal-plan-{timestamp}"

    os.makedirs(folder, exist_ok=True)

    result = f"""
# Weekly Meal Plan

{meal_plan}

# Grocery List

{grocery_list}

# Review

{review}
"""

    with open(
        f"{folder}/result.md",
        "w",
        encoding="utf-8"
    ) as f:

        f.write(result)

    run_data = {
        "diet": diet,
        "people": people,
        "cuisine": cuisine,
        "requirements": requirements,
        "meal_plan": meal_plan,
        "grocery_list": grocery_list,
        "review": review
    }

    with open(
        f"{folder}/run.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            run_data,
            f,
            indent=2
        )

    print("Saved:", folder)

    return result

In [12]:
print(
    analyze_requirements(
        "Vegetarian",
        4,
        "Indian"
    )
)

{
  "requestSummary": {
    "diet": "Vegetarian",
    "cuisine": "Indian",
    "people": 4
  },
  "derived": {
    "servingsPerMeal": 4
  },
  "planScope": {
    "planLengthDays": null,
    "mealsPerDay": null,
    "mealTypes": [],
    "coursesPerMeal": null,
    "includeSnacks": null,
    "occasion": null
  },
  "nutrition": {
    "caloriesPerPersonPerDay": null,
    "proteinPerPersonPerDay": null,
    "carbPreference": null,
    "fatPreference": null,
    "fiberTarget": null
  },
  "dietaryDetails": {
    "dairyOk": null,
    "eggOk": null,
    "onionGarlicOk": null,
    "rootVegetablesOk": null,
    "glutenFree": null,
    "soyOk": null,
    "nutsOk": null
  },
  "tasteAndSpice": {
    "spiceLevel": null,
    "heatToleranceNotes": null,
    "flavorPreferences": []
  },
  "constraints": {
    "allergies": [],
    "exclusions": [],
    "preferredIngredients": [],
    "seasonalFocus": null,
    "religiousOrCulturalRules": null
  },
  "timeAndEffort": {
    "prepTimeMaxMinutes": null,
 

# Empirical Testing

## Test Runs

| Criteria | Run 1 | Run 2 | Run 3 |
|-----------|--------|--------|--------|
| Follows Diet | Pass | Pass | Pass |
| No Repeats | Pass | Fail | Pass |
| Correct Portions | Pass | Pass | Pass |
| Grocery List Merged | Pass | Pass | Pass |
| 7 Days Covered | Pass | Pass | Pass |

## Reflection

During testing, one run repeated a meal within 3 days. The meal planner prompt was updated to emphasize meal variety and avoid repetition. After the change, later runs satisfied all criteria more consistently.

In [18]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

folder = f"outputs/meal-plan-{timestamp}"

print(folder)

outputs/meal-plan-20260620-194157


In [19]:
from datetime import datetime
import os
import json